In [1]:
from pyspark.sql import SparkSession

from pyspark.sql.functions import (
    sum,
    when,
    col,
    avg
)

from pyspark.sql.window import Window

import os
import time

from datetime import datetime

In [12]:
# ==========================================
# HDFS PATHS
# ==========================================

HDFS_BASE = "hdfs://master:9000"

RAW_INPUT_PATH = (
    f"{HDFS_BASE}/project/input/raw/uncomtrade"
)

STAGE1_OUTPUT = (
    f"{HDFS_BASE}/project/tmp/uncomtrade/stage1_country_year_totals"
)

STAGE2_OUTPUT = (
    f"{HDFS_BASE}/project/tmp/uncomtrade/stage2_country_year_shares"
)

STAGE3_OUTPUT = (
    f"{HDFS_BASE}/project/tmp/uncomtrade/stage3_country_year_future_avg"
)

# ==========================================
# LOGGING
# ==========================================

LOG_FILE = "/shared/logs/pipeline.log"

os.makedirs("/shared/logs", exist_ok=True)

# ==========================================
# PRODUCT CODES
# ==========================================

FUEL_CODE = "27"
GRAIN_CODE = "12"
GUN_CODE = "93"

In [13]:
def log_message(stage, status, message):

    timestamp = datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

    log_line = (
        f"[{timestamp}] "
        f"STAGE={stage} "
        f"STATUS={status} "
        f"MESSAGE={message}"
    )

    print(log_line)

    with open(LOG_FILE, "a") as f:
        f.write(log_line + "\n")

In [14]:
def preview_parquet(path, rows=10):

    spark.read.parquet(path).show(
        rows,
        truncate=False
    )

In [15]:
spark = (
    SparkSession.builder
    .appName("UN_Comtrade_ETL")
    .getOrCreate()
)

In [16]:
stage_name = "STAGE1"

start_time = time.time()

try:

    log_message(
        stage_name,
        "STARTED",
        f"INPUT={RAW_INPUT_PATH}"
    )

    input_df = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(RAW_INPUT_PATH)
        .select(
            "reporterISO",
            "refYear",
            "cmdCode",
            "primaryValue"
        )
    )

    stage1_df = (
        input_df

        .groupBy(
            "reporterISO",
            "refYear"
        )

        .agg(

            sum("primaryValue")
            .alias("trade_total"),

            sum(
                when(
                    col("cmdCode") == FUEL_CODE,
                    col("primaryValue")
                ).otherwise(0)
            ).alias("fuel_value"),

            sum(
                when(
                    col("cmdCode") == GRAIN_CODE,
                    col("primaryValue")
                ).otherwise(0)
            ).alias("grain_value"),

            sum(
                when(
                    col("cmdCode") == GUN_CODE,
                    col("primaryValue")
                ).otherwise(0)
            ).alias("gun_value")
        )

        .orderBy(
            "reporterISO",
            "refYear"
        )
    )

    (
        stage1_df.write
        .mode("overwrite")
        .parquet(STAGE1_OUTPUT)
    )

    elapsed = time.time() - start_time

    log_message(
        stage_name,
        "SUCCESS",
        f"Completed in {elapsed:.2f} seconds"
    )

except Exception as e:

    elapsed = time.time() - start_time

    log_message(
        stage_name,
        "FAILED",
        f"Error after {elapsed:.2f} seconds: {str(e)}"
    )

    raise

[2026-05-26 22:54:17] STAGE=STAGE1 STATUS=STARTED MESSAGE=INPUT=hdfs://master:9000/project/input/raw/uncomtrade


[2026-05-26 23:06:22] STAGE=STAGE1 STATUS=SUCCESS MESSAGE=Completed in 724.73 seconds


In [17]:
preview_parquet(STAGE1_OUTPUT)

+-----------+-------+-------------+------------+-----------+---------+
|reporterISO|refYear|trade_total  |fuel_value  |grain_value|gun_value|
+-----------+-------+-------------+------------+-----------+---------+
|ABW        |2000   |1.916868138E9|0.0         |1062381.0  |0.0      |
|ABW        |2001   |1.834571823E9|0.0         |916816.0   |238581.0 |
|ABW        |2002   |1.783732832E9|0.0         |1134627.0  |325040.0 |
|ABW        |2003   |9.62870126E8 |0.0         |0.0        |201833.0 |
|ABW        |2004   |1.88187764E9 |0.0         |738951.0   |272692.0 |
|ABW        |2005   |2.242634832E9|0.0         |590704.0   |239972.0 |
|ABW        |2006   |2.264211364E9|0.0         |693232.0   |292112.0 |
|ABW        |2007   |2.388771304E9|0.0         |1066109.0  |246060.0 |
|ABW        |2008   |2.388069474E9|0.0         |1355743.0  |100079.0 |
|ABW        |2009   |2.540678815E9|1.17615733E8|1256577.0  |368594.0 |
+-----------+-------+-------------+------------+-----------+---------+
only s

In [18]:
stage_name = "STAGE2"

start_time = time.time()

try:

    log_message(
        stage_name,
        "STARTED",
        f"INPUT={STAGE1_OUTPUT}"
    )

    stage1_df = spark.read.parquet(
        STAGE1_OUTPUT
    )

    stage2_df = (

        stage1_df

        .withColumn(
            "fuel_share",

            when(
                col("trade_total") > 0,

                col("fuel_value")
                / col("trade_total")

            ).otherwise(0)
        )

        .withColumn(
            "grain_share",

            when(
                col("trade_total") > 0,

                col("grain_value")
                / col("trade_total")

            ).otherwise(0)
        )

        .withColumn(
            "gun_share",

            when(
                col("trade_total") > 0,

                col("gun_value")
                / col("trade_total")

            ).otherwise(0)
        )

        .select(

            col("reporterISO")
            .alias("iso3"),

            col("refYear")
            .alias("year"),

            "fuel_share",
            "grain_share",
            "gun_share"
        )

        .orderBy(
            "iso3",
            "year"
        )
    )

    (
        stage2_df.write
        .mode("overwrite")
        .parquet(STAGE2_OUTPUT)
    )

    elapsed = time.time() - start_time

    log_message(
        stage_name,
        "SUCCESS",
        f"Completed in {elapsed:.2f} seconds"
    )

except Exception as e:

    elapsed = time.time() - start_time

    log_message(
        stage_name,
        "FAILED",
        f"Error after {elapsed:.2f} seconds: {str(e)}"
    )

    raise

[2026-05-26 23:19:34] STAGE=STAGE2 STATUS=STARTED MESSAGE=INPUT=hdfs://master:9000/project/tmp/uncomtrade/stage1_country_year_totals


[2026-05-26 23:19:36] STAGE=STAGE2 STATUS=SUCCESS MESSAGE=Completed in 1.73 seconds


In [19]:
preview_parquet(STAGE2_OUTPUT)

+----+----+-------------------+---------------------+---------------------+
|iso3|year|fuel_share         |grain_share          |gun_share            |
+----+----+-------------------+---------------------+---------------------+
|ABW |2000|0.0                |5.542274812436785E-4 |0.0                  |
|ABW |2001|0.0                |4.997438576707062E-4 |1.3004723882102271E-4|
|ABW |2002|0.0                |6.360969421232249E-4 |1.822245989807514E-4 |
|ABW |2003|0.0                |0.0                  |2.0961601627258295E-4|
|ABW |2004|0.0                |3.9266686860682396E-4|1.4490421385738978E-4|
|ABW |2005|0.0                |2.633973179990277E-4 |1.070044915809994E-4 |
|ABW |2006|0.0                |3.06169296304265E-4  |1.2901269053077678E-4|
|ABW |2007|0.0                |4.4630015364585104E-4|1.0300693062913652E-4|
|ABW |2008|0.0                |5.677150580251502E-4 |4.1907909752880163E-5|
|ABW |2009|0.04629303487934188|4.945831769766617E-4 |1.4507697620960404E-4|
+----+----+-

In [20]:
stage_name = "STAGE3"

start_time = time.time()

try:

    log_message(
        stage_name,
        "STARTED",
        f"INPUT={STAGE2_OUTPUT}"
    )

    stage2_df = spark.read.parquet(
        STAGE2_OUTPUT
    )

    future_window = (

        Window

        .partitionBy("iso3")

        .orderBy("year")

        .rowsBetween(1, 5)
    )

    stage3_df = (

        stage2_df

        .withColumn(
            "avg_fuel_next5y",

            avg("fuel_share")
            .over(future_window)
        )

        .withColumn(
            "avg_grain_next5y",

            avg("grain_share")
            .over(future_window)
        )

        .withColumn(
            "avg_gun_next5y",

            avg("gun_share")
            .over(future_window)
        )

        .orderBy(
            "iso3",
            "year"
        )
    )

    (
        stage3_df.write
        .mode("overwrite")
        .parquet(STAGE3_OUTPUT)
    )

    elapsed = time.time() - start_time

    log_message(
        stage_name,
        "SUCCESS",
        f"Completed in {elapsed:.2f} seconds"
    )

except Exception as e:

    elapsed = time.time() - start_time

    log_message(
        stage_name,
        "FAILED",
        f"Error after {elapsed:.2f} seconds: {str(e)}"
    )

    raise

[2026-05-26 23:19:44] STAGE=STAGE3 STATUS=STARTED MESSAGE=INPUT=hdfs://master:9000/project/tmp/uncomtrade/stage2_country_year_shares


[2026-05-26 23:19:46] STAGE=STAGE3 STATUS=SUCCESS MESSAGE=Completed in 1.61 seconds


In [21]:
preview_parquet(STAGE3_OUTPUT)

+----+----+-------------------+---------------------+---------------------+--------------------+---------------------+---------------------+
|iso3|year|fuel_share         |grain_share          |gun_share            |avg_fuel_next5y     |avg_grain_next5y     |avg_gun_next5y       |
+----+----+-------------------+---------------------+---------------------+--------------------+---------------------+---------------------+
|ABW |2000|0.0                |5.542274812436785E-4 |0.0                  |0.0                 |3.583809972799565E-4 |1.5475931190254925E-4|
|ABW |2001|0.0                |4.997438576707062E-4 |1.3004723882102271E-4|0.0                 |3.196660850066683E-4 |1.5455240224450007E-4|
|ABW |2002|0.0                |6.360969421232249E-4 |1.822245989807514E-4 |0.0                 |2.8170672731119353E-4|1.387088685741771E-4 |
|ABW |2003|0.0                |0.0                  |2.0961601627258295E-4|0.0                 |3.9524973891622353E-4|1.0516724727023653E-4|
|ABW |2004|0.